In [3]:
import tensorflow as tf
from tensorflow.keras import layers, Model

class GPT(Model):
    def __init__(self, vocab_size, block_size, embed_dim=128, num_heads=4, num_layers=2):
        super().__init__()

        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = layers.Embedding(block_size, embed_dim)

        self.blocks = [
            TransformerBlock(embed_dim, num_heads)
            for _ in range(num_layers)
        ]

        self.ln_f = layers.LayerNormalization()
        self.lm_head = layers.Dense(vocab_size)

        self.block_size = block_size

    def call(self, x):
        B, T = tf.shape(x)[0], tf.shape(x)[1]

        tok = self.token_emb(x)
        pos = self.pos_emb(tf.range(T)[tf.newaxis, :])
        h = tok + pos

        for block in self.blocks:
            h = block(h)

        h = self.ln_f(h)
        return self.lm_head(h)





In [4]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.ln1 = layers.LayerNormalization()
        self.ln2 = layers.LayerNormalization()

        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )

        self.ff = tf.keras.Sequential([
            layers.Dense(4 * embed_dim, activation='relu'),
            layers.Dense(embed_dim),
        ])

    def call(self, x):
        T = tf.shape(x)[1]

        # manual causal mask (Lower triangular)
        causal_mask = tf.linalg.band_part(tf.ones((T, T)), -1, 0)
        causal_mask = causal_mask[tf.newaxis, tf.newaxis, :, :]  # (1,1,T,T)

        att_out = self.att(
            self.ln1(x),
            self.ln1(x),
            attention_mask=causal_mask
        )

        x = x + att_out
        x = x + self.ff(self.ln2(x))
        return x


In [5]:
import numpy as np

text = """
once upon a time there was a brave knight.
the knight fought dragons and saved villages.
one day a dragon appeared near the mountains.
the knight prepared for the battle.
villagers watched as the sun slowly set.
"""

chars = sorted(list(set(train_text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s): return [stoi[c] for c in s]
def decode(t): return "".join([itos[i] for i in t])

data = encode(text)
data = np.array(data, dtype=np.int32)


NameError: name 'train_text' is not defined

In [6]:
block_size = 32  # context size

def make_dataset(data, block_size):
    xs = []
    ys = []
    for i in range(len(data) - block_size):
        x = data[i:i+block_size]
        y = data[i+1:i+block_size+1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

# x_train, y_train = make_dataset(data, block_size)


In [13]:
import tensorflow as tf

batch_size = 32

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(1000).batch(batch_size).prefetch(1)


In [14]:
model = GPT(vocab_size, block_size)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

model.fit(train_ds, epochs=20)



Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 70ms/step - loss: 2.8880
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 1.8546
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 1.3080
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - loss: 0.9958
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 0.7491
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 0.5517
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 0.4026
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step - loss: 0.3006 
Epoch 9/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - loss: 0.2275
Epoch 10/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - loss: 0.1797
Epoch 11/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - loss: 0.1551
Epoch 12/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 99ms/step - loss: 0.1429 
Epoch 13/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - loss: 0.1278
Epoch 14/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - loss: 0.1201
Epoch 15/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 0.1174
Epoch 16/20
6/6 ━━━━━━━━━━━━━━━━

In [15]:
def generate(model, start_token, num_tokens=100):
    idx = [stoi[start_token]]

    for _ in range(num_tokens):
        x = tf.constant([idx[-block_size:]], dtype=tf.int32)

        logits = model(x)
        next_id = tf.random.categorical(logits[:, -1, :], 1)[0][0].numpy()

        idx.append(next_id)

    return decode(idx)


In [16]:
print(generate(model, "m", 200))


me the was a brave knight.
the knight fought dragons and saveved villages.
one day a dragon appeared near the mountains.
the knight prepared for the battle.
villagers watched as the sun slowly set.
vil


In [7]:
qa_pairs = [
    ("What is machine learning?", "Machine learning is a method where computers learn from data."),
    ("What is AI?", "AI is the simulation of human intelligence in machines."),
    ("What is deep learning?", "Deep learning uses neural networks with multiple layers."),
    ("What is a model?", "A model is a mathematical system trained to make predictions."),
    ("What is training data?", "Training data is labeled information used to teach a model."),
    ("What is a GPU?", "A GPU is hardware optimized for parallel mathematical operations."),
    ("What is a neuron?", "A neuron is a unit inside a neural network that processes data."),
    ("What is backpropagation?", "Backpropagation is an algorithm used to train neural networks."),
    ("What is gradient descent?", "Gradient descent is an optimization method for minimizing loss."),
    ("What is overfitting?", "Overfitting happens when a model memorizes training data."),
    ("What is underfitting?", "Underfitting means the model is too simple."),
    ("What is regularization?", "Regularization reduces overfitting by adding constraints."),
    ("What is dropout?", "Dropout randomly disables neurons during training."),
    ("What is a loss function?", "A loss function measures the error of predictions."),
    ("What is an epoch?", "An epoch is one full training pass over the dataset."),
    ("What is accuracy?", "Accuracy measures how many predictions are correct."),
    ("What is a dataset?", "A dataset is a structured collection of data."),
    ("What is reinforcement learning?", "Reinforcement learning trains agents using rewards."),
    ("What is NLP?", "NLP is the field of processing human language."),
    ("What is a token?", "A token is the smallest unit of text in NLP."),
    ("What is BERT?", "BERT is a transformer model for understanding language."),
    ("What is GPT?", "GPT is a transformer-based model for generating text."),
    ("What is attention?", "Attention helps the model focus on important parts of input."),
    ("What is a transformer?", "A transformer is a model based on self-attention mechanisms."),
    ("What is a batch?", "A batch is a group of samples processed together."),
]


In [8]:

# Create training text
train_text = ""
for q, a in qa_pairs:
    train_text += f"Q: {q}\nA: {a}\n\n"


In [9]:

chars = sorted(list(set(train_text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

In [ ]:


def encode(s):
    return np.array([stoi[c] for c in s], dtype=np.int32)

def decode(tokens):
    return "".join([itos[t] for t in tokens])

data = encode(train_text)


block_size = 64  # context length

def make_dataset(data, block_size):
    xs, ys = [], []
    for i in range(len(data) - block_size):
        xs.append(data[i:i+block_size])
        ys.append(data[i+1:i+block_size+1])
    return np.array(xs), np.array(ys)

x_train, y_train = make_dataset(data, block_size)

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(1000).batch(32)

In [11]:
model = GPT(vocab_size, block_size)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

model.fit(train_ds, epochs=15)

Epoch 1/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 18s 159ms/step - loss: 2.1276
Epoch 2/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 139ms/step - loss: 1.2327
Epoch 3/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 151ms/step - loss: 0.4351
Epoch 4/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 145ms/step - loss: 0.1840
Epoch 5/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 131ms/step - loss: 0.1408
Epoch 6/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 127ms/step - loss: 0.1344
Epoch 7/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 124ms/step - loss: 0.1167
Epoch 8/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 128ms/step - loss: 0.1155
Epoch 9/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 125ms/step - loss: 0.1077
Epoch 10/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 121ms/step - loss: 0.1075
Epoch 11/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 126ms/step - loss: 0.1036
Epoch 12/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 123ms/step - loss: 0.1031
Epoch 13/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 152ms/step - loss: 0.1013
Epoch 14/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 136ms/step - loss: 0.1038
Epoch 15/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 161ms/step 

In [12]:
def generate(model, start, num_tokens=200):
    idx = [stoi[c] for c in start]

    for _ in range(num_tokens):
        x = np.array(idx[-block_size:], dtype=np.int32)[np.newaxis, :]
        logits = model(x)
        next_id = tf.random.categorical(logits[:, -1, :], 1).numpy()[0, 0]
        idx.append(next_id)

    return decode(idx)

In [19]:
print(generate(model, "Q: What is AI", 70))

Q: What is AI?
A: AI is the simulation of human intelligence in machines.

Q: What 
